# SAGE — Bằng chứng nghiên cứu cho báo cáo 08/11/2026 (Lộc bé)

**Mục tiêu khác báo cáo 05/11:** kiểm thử cùng **10 cặp FMB** đã chạy SAGE; đối chứng hai kỹ thuật fusion cố định; đo metric và xem các failure case trước khi đề xuất phương pháp.

- **Đã có thật**: 10 ảnh fused của SAGE trong `SAGE_LocBe_pretrained_inference_results.zip`.
- **Cần có trên Kaggle**: 10 fused PNG gốc (`SAGE_LocBe_pretrained_inference_results.zip` gắn qua Add Input hoặc thư mục `fused_png`); TAR FMB `fmb-test-00000.tar` có thể lấy từ Input/cache hoặc tự tải Hugging Face nếu bật Internet.
- **Đối chứng trong notebook:** `Y_AVG` (trung bình Y-visible, IR) và `Y_MAX` (max pixel theo Y), **không phải các model SOTA**.
- **Metric:** EN/SD, SCD (correlation differences), MS-SSIM hai nguồn cộng, exploratory gradient correlation và IR-hot contrast ratio.
- **Không khẳng định** các metric bằng chính xác triển khai của paper SAGE nếu chưa so code/trọng số/channels; không dùng FMB/test để chọn tham số hoặc làm training.

Chạy tuần tự toàn bộ notebook trong Kaggle; bật Internet nếu cần cài `pytorch-msssim`. Nếu GPU không có, metric vẫn chạy bằng CPU (chậm hơn).

> **Lưu ý:** Phiên Kaggle mới không tự nhìn thấy `/kaggle/working` của notebook cũ. Nếu bạn dùng Notebook mới, hãy Add Input output của notebook cũ hoặc tải ZIP fused vào một Kaggle Dataset và Add Input. TAR FMB có thể tự tải từ Hugging Face (cần Internet). GPU giúp chạy MS-SSIM nhanh nhưng không chạy model SAGE.


In [ ]:
from pathlib import Path
import os, io, tarfile, zipfile, json, csv, hashlib, subprocess, sys, platform
from datetime import datetime, timezone
import numpy as np, cv2, pandas as pd, torch
from PIL import Image
import matplotlib.pyplot as plt

IDS = ['00014','00024','00061','00064','00071','00075','00104','00109','00112','00145']
BASE = Path('/kaggle/working')
RUN_ROOT = BASE / 'sage_08nov_laybe_diag'
if RUN_ROOT.exists():
    # Các lần chạy lặp lại giữ file cũ ở chỗ cũ, không ghi đè.
    RUN_ROOT = BASE / ('sage_08nov_laybe_diag_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'))
RUN_ROOT.mkdir(parents=True, exist_ok=False)
(RUN_ROOT / 'figures').mkdir()
(RUN_ROOT / 'metrics').mkdir()

print('RUN_ROOT:', RUN_ROOT)
print('Python:', sys.version.split()[0], 'Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


## 1. Tìm đúng ảnh IR/VIS và fused trước khi tính metric

File TAR FMB WebDataset có `00014.ir.png` và `00014.rgb.png`; **ID phải khớp**, không ghép bằng thứ tự của thư mục. Nguồn ảnh của bản FMB Hugging Face repackaged chưa được chứng minh cùng split/kích thước với FMB chính thức. Không dùng số liệu này để so trực tiếp các con số từ paper.

Nếu bạn dùng Notebook mới: **Add Input → Notebooks → chọn output của notebook cũ** (phải có một phiên bản lưu output) hoặc **Add Input → Datasets → chọn dataset chứa ZIP fused**. Không cần gắn checkpoint.


In [ ]:
# 1) Tìm TAR trong cache hiện tại và các nguồn Add Input (read-only).
TAR_CANDIDATES = sorted(BASE.rglob('fmb-test-00000.tar'))
TAR_CANDIDATES += sorted(Path('/kaggle/input').rglob('fmb-test-00000.tar'))

# 2) Nếu là Notebook mới, tự tải đúng TAR FMB đã dùng ngày 05/11.
if not TAR_CANDIDATES:
    print('Chưa có TAR trong working/input; thử tải FMB/test từ Hugging Face...')
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
        from huggingface_hub import hf_hub_download
    try:
        downloaded = hf_hub_download(
            repo_id='LC-State/RGB2IR_Translation', repo_type='dataset',
            filename='data/FMB/test/fmb-test-00000.tar',
            local_dir=str(BASE / 'sage_hf_cache'))
        TAR_CANDIDATES = [Path(downloaded)]
    except Exception as e:
        raise RuntimeError(
            'Không lấy được TAR FMB. Bật Internet trong Kaggle Session options '
            'hoặc Add Input dữ liệu FMB chứa file fmb-test-00000.tar. '
            f'Chi tiết: {type(e).__name__}: {e}') from e
TAR_FILE = TAR_CANDIDATES[0]
print('TAR:', TAR_FILE)

def img_from_bytes(data, mode):
    with Image.open(io.BytesIO(data)) as im:
        return np.array(im.convert(mode))

with tarfile.open(TAR_FILE, 'r:*') as tf:
    members = {m.name.rsplit('/',1)[-1]: m for m in tf if m.isfile()}
    assert all(f'{sid}.rgb.png' in members and f'{sid}.ir.png' in members for sid in IDS), \
        'TAR thiếu ít nhất một cặp IR–VIS của 10 ID đã chọn.'
    SOURCES = {}
    for sid in IDS:
        vis_b = tf.extractfile(members[f'{sid}.rgb.png']).read()
        ir_b = tf.extractfile(members[f'{sid}.ir.png']).read()
        vis = img_from_bytes(vis_b, 'RGB')
        ir = img_from_bytes(ir_b, 'L')
        assert vis.shape[:2] == ir.shape, f'Kích thước không khớp: {sid}'
        SOURCES[sid] = {'vis':vis, 'ir':ir, 'vis_sha256':hashlib.sha256(vis_b).hexdigest(),
                        'ir_sha256':hashlib.sha256(ir_b).hexdigest()}
print('Đã đọc nguồn IR-VIS:', len(SOURCES), 'cặp; shape:', SOURCES[IDS[0]]['ir'].shape)


In [ ]:
# Tìm bộ PNG đã chạy SAGE trước đó; hỗ trợ file đã bung và ZIP đính kèm trên Kaggle.
# Không chọn ảnh IR/VIS nhầm làm fused.
FUSED_FOLDER_CANDIDATES = [
    BASE / 'sage_locbe_outputs' / 'fused_png',
    BASE / 'SAGE_LocBe_pretrained_inference_results' / 'fused_png',
]
FUSED_FOLDER_CANDIDATES += [p for p in BASE.rglob('fused_png') if p.is_dir()]
FUSED_FOLDER_CANDIDATES += [p for p in Path('/kaggle/input').rglob('fused_png') if p.is_dir()]

def has_all_pngs(p):
    return p.is_dir() and all((p / f'{sid}.png').is_file() for sid in IDS)

folder = next((p for p in FUSED_FOLDER_CANDIDATES if has_all_pngs(p)), None)
FUSED = {}
FUSED_SOURCE = None
if folder:
    FUSED_SOURCE = str(folder)
    for sid in IDS:
        FUSED[sid] = img_from_bytes((folder / f'{sid}.png').read_bytes(), 'RGB')
else:
    zip_candidates = sorted(set(BASE.rglob('SAGE_LocBe_pretrained_inference_results.zip')))
    zip_candidates += sorted(set(Path('/kaggle/input').rglob('SAGE_LocBe_pretrained_inference_results.zip')))
    for zp in zip_candidates:
        with zipfile.ZipFile(zp) as z:
            names = set(z.namelist())
            if all(f'fused_png/{sid}.png' in names for sid in IDS):
                FUSED_SOURCE = str(zp)
                for sid in IDS:
                    FUSED[sid] = img_from_bytes(z.read(f'fused_png/{sid}.png'), 'RGB')
                break

assert len(FUSED) == 10, ('Chưa thấy đủ 10 ảnh fused SAGE trong /kaggle/working hoặc /kaggle/input. '
                          'Hãy Add Input file ZIP đầu ra đã tạo hoặc đặt fused_png đúng đường dẫn.')
for sid in IDS:
    assert FUSED[sid].shape == SOURCES[sid]['vis'].shape, f'Output SAGE sai kích thước: {sid}'
print('SAGE fused source:', FUSED_SOURCE)
print('Đã xác thực:', len(FUSED), 'output cùng kích thước và ID với 10 cặp IR–VIS.')


## 2. Ba phương pháp được đánh giá trên cùng 10 ID

**SAGE:** ảnh đã chạy pretrained Student, không tạo mới hay chỉnh sửa. **Y_AVG:** \(Y=0.5Y_{VIS}+0.5IR\). **Y_MAX:** \(Y=\max(Y_{VIS},IR)\). Hai phương pháp sau chỉ là đối chứng đơn giản, không phải SOTA và không được dùng để tuyên bố đóng góp mới.

In [ ]:
def visible_y(vis_rgb):
    return cv2.cvtColor(vis_rgb, cv2.COLOR_RGB2YCrCb)[:,:,0]

def reconstruct_rgb(vis_rgb, target_y):
    ycc = cv2.cvtColor(vis_rgb, cv2.COLOR_RGB2YCrCb).copy()
    ycc[:,:,0] = np.asarray(np.clip(np.rint(target_y),0,255),dtype=np.uint8)
    return cv2.cvtColor(ycc, cv2.COLOR_YCrCb2RGB)

OUTPUTS = {'SAGE':FUSED.copy(), 'Y_AVG':{}, 'Y_MAX':{}}
for sid in IDS:
    src = SOURCES[sid]
    y_vis = visible_y(src['vis']).astype(np.float64)
    ir = src['ir'].astype(np.float64)
    OUTPUTS['Y_AVG'][sid] = reconstruct_rgb(src['vis'], (y_vis+ir)*0.5)
    OUTPUTS['Y_MAX'][sid] = reconstruct_rgb(src['vis'], np.maximum(y_vis,ir))
    for method in ['Y_AVG','Y_MAX']:
        assert OUTPUTS[method][sid].shape==FUSED[sid].shape
print('Các method:',list(OUTPUTS),'— cùng',len(IDS),'ảnh/ method.')


## 3. Tính các metric và lưu công thức

SCD: `corr(F − IR, VIS) + corr(F − VIS, IR)` theo định nghĩa correlation of differences. MS-SSIM: cộng `MS-SSIM(F,IR)` và `MS-SSIM(F,VIS)` (không phải SSIM thường). Các ảnh đánh giá đều đưa về kênh xám; EN/SD ở thang 0–255; MS-SSIM ở [0,1].

`IR_hot_ratio` = độ tương phản giữa nhóm top 10% độ sáng IR và phần còn lại *trên fused* chia cho độ tương phản tương ứng *trên IR*; chỉ là **proxy exploratory**, không phải đo segmentation/semantic target.

In [ ]:
try:
    from pytorch_msssim import ms_ssim
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','pytorch-msssim'],check=True)
    from pytorch_msssim import ms_ssim

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
def gray(rgb):
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

def entropy(g):
    hist=np.bincount(g.reshape(-1),minlength=256).astype(np.float64)
    p=hist[hist>0]/hist.sum()
    return float(-(p*np.log2(p)).sum())

def pearson(a,b):
    a=np.asarray(a,dtype=np.float64).ravel().copy(); b=np.asarray(b,dtype=np.float64).ravel().copy()
    a-=a.mean(); b-=b.mean()
    den=np.linalg.norm(a)*np.linalg.norm(b)
    return float((a@b)/den) if den>1e-12 else float('nan')

def scd(f,ir,vis):
    f=f.astype(np.float64); ir=ir.astype(np.float64); vis=vis.astype(np.float64)
    return pearson(f-ir,vis)+pearson(f-vis,ir)

def mssim_pair(a,b):
    x=torch.from_numpy(np.ascontiguousarray(a/255.0)).float()[None,None].to(DEVICE)
    y=torch.from_numpy(np.ascontiguousarray(b/255.0)).float()[None,None].to(DEVICE)
    with torch.inference_mode():
        return float(ms_ssim(x,y,data_range=1.,size_average=True).item())

def edge_corr(a,b):
    def mag(t):
        t=np.asarray(t,dtype=np.float64)
        return np.hypot(cv2.Sobel(t,cv2.CV_64F,1,0,ksize=3),
                        cv2.Sobel(t,cv2.CV_64F,0,1,ksize=3))
    return pearson(mag(a),mag(b))

def ir_hot_ratio(f,ir):
    hot=ir>=np.quantile(ir,0.90)
    assert hot.any() and (~hot).any(), 'Hot-region mask không hợp lệ'
    ir_contrast=float(ir[hot].mean()-ir[~hot].mean())
    f_contrast=float(f[hot].mean()-f[~hot].mean())
    return f_contrast/ir_contrast if abs(ir_contrast)>1e-8 else float('nan')

# Smoke kiểm định SCD bất biến khi cộng hằng số vào F và có thể âm; tránh uint8 wrap-around.
rng=np.random.default_rng(42)
a=rng.integers(0,256,size=(32,32),dtype=np.uint8)
b=rng.integers(0,256,size=(32,32),dtype=np.uint8)
f=(a.astype(float)+b.astype(float))*0.5
assert np.isclose(scd(f,a,b),scd(f+4,a,b),atol=1e-9)
assert np.isfinite(scd(f,a,b))
print('SCD sanity pass; ms-ssim backend:',DEVICE)


In [ ]:
records=[]
for sid in IDS:
    ir=SOURCES[sid]['ir']; vi=visible_y(SOURCES[sid]['vis'])
    for method, imgs in OUTPUTS.items():
        fg=gray(imgs[sid]); mf=mssim_pair(fg,ir); mv=mssim_pair(fg,vi)
        records.append({'image_id':sid,'method':method,
                        'EN':entropy(fg),'SD':float(fg.astype(np.float64).std(ddof=0)),
                        'SCD':scd(fg,ir,vi), 'MS_SSIM_IR':mf, 'MS_SSIM_VIS':mv,
                        'MS_SSIM_sum':mf+mv,
                        'edge_corr_IR':edge_corr(fg,ir),
                        'edge_corr_VIS':edge_corr(fg,vi),
                        'IR_hot_ratio':ir_hot_ratio(fg.astype(float),ir.astype(float)),
                        'valid':True,'error_reason':''})

per=pd.DataFrame(records)
assert len(per)==30 and all(per.groupby('method').size()==10)
per.to_csv(RUN_ROOT/'metrics'/'per_image.csv',index=False)
metrics=['EN','SD','SCD','MS_SSIM_sum','edge_corr_IR','edge_corr_VIS','IR_hot_ratio']
summary=per.groupby('method')[metrics].agg(['mean','std','median'])
summary.to_csv(RUN_ROOT/'metrics'/'summary.csv')
display(summary.round(4))


In [ ]:
# Chênh lệch theo cặp: cùng image_id, SAGE trừ từng đối chứng.
paired=[]
for sid in IDS:
    base=per[(per.image_id==sid)&(per.method=='SAGE')].iloc[0]
    for comp in ['Y_AVG','Y_MAX']:
        other=per[(per.image_id==sid)&(per.method==comp)].iloc[0]
        paired.append({'image_id':sid,'comparison':f'SAGE_minus_{comp}',
                       **{f'delta_{col}':float(base[col]-other[col]) for col in metrics}})
paired=pd.DataFrame(paired)
paired.to_csv(RUN_ROOT/'metrics'/'paired_comparison.csv',index=False)

# Trường hợp cần xem: SAGE có IR_hot_ratio thấp hơn Y_AVG.
focus=paired[paired.comparison=='SAGE_minus_Y_AVG'].sort_values('delta_IR_hot_ratio')
print('3 ảnh đáng xem trước:',focus.image_id.head(3).tolist())
display(focus[['image_id','delta_SCD','delta_MS_SSIM_sum','delta_IR_hot_ratio']].round(4))


In [ ]:
# Vẽ hình cho 3 ID cần xem; MỘT tập ảnh dùng cho cả ba method.
selected=focus.image_id.head(3).tolist()
fig,axes=plt.subplots(len(selected),5,figsize=(16,3.2*len(selected)),squeeze=False)
for i,sid in enumerate(selected):
    panels=[SOURCES[sid]['ir'],SOURCES[sid]['vis'],OUTPUTS['SAGE'][sid],
            OUTPUTS['Y_AVG'][sid],OUTPUTS['Y_MAX'][sid]]
    for j,(name,pic) in enumerate(zip(['IR','VIS','SAGE','Y_AVG','Y_MAX'],panels)):
        axes[i,j].imshow(pic,cmap='gray' if j==0 else None)
        axes[i,j].set_title(f'{sid}: {name}',fontsize=10)
        axes[i,j].axis('off')
plt.tight_layout()
fig.savefig(RUN_ROOT/'figures'/'failure_case_candidates.png',dpi=120,bbox_inches='tight')
plt.show()


In [ ]:
# Lưu provenance, manifest và báo cáo tự sinh; KHÔNG tự kết luận phương pháp đã tốt hơn.
manifest=[]
for sid in IDS:
    manifest.append({'image_id':sid,'split':'FMB/test_repackaged',
                     'ir_member':f'{sid}.ir.png','vis_member':f'{sid}.rgb.png',
                     'height':int(SOURCES[sid]['ir'].shape[0]),
                     'width':int(SOURCES[sid]['ir'].shape[1]),
                     'ir_sha256':SOURCES[sid]['ir_sha256'],
                     'vis_sha256':SOURCES[sid]['vis_sha256']})
pd.DataFrame(manifest).to_csv(RUN_ROOT/'manifest.csv',index=False)
config={'run_id':RUN_ROOT.name,'date_utc':datetime.now(timezone.utc).isoformat(),
        'experiment':'SAGE vs Y_AVG vs Y_MAX; 10 paired IR/VIS samples',
        'image_ids':IDS,'dataset_archive':str(TAR_FILE),
        'source_repackaged':'LC-State/RGB2IR_Translation; FMB/test',
        'sage_fused_location':FUSED_SOURCE,
        'metrics_definitions':{'EN':'Shannon 256 bins on RGB2GRAY 8bit',
            'SD':'population std RGB2GRAY 0..255',
            'SCD':'corr(F-IR,VIS_Y)+corr(F-VIS_Y,IR)',
            'MS_SSIM_sum':'pytorch_msssim.ms_ssim(F,IR)+ms_ssim(F,VIS_Y), data_range=1',
            'IR_hot_ratio':'top IR intensity decile contrast retention, proxy only'},
        'limitations':['NOT official FMB full-test 280 pair reproduction',
                       'SOTA models not executed',
                       'No segmentation/mIoU',
                       'Metric definitions require owner review for paper parity',
                       'This test set must NOT be used to tune hyperparameters']}
(RUN_ROOT/'run_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
(RUN_ROOT/'environment.json').write_text(json.dumps({'python':sys.version,'torch':torch.__version__,
  'opencv':cv2.__version__,'numpy':np.__version__,
  'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'},indent=2),encoding='utf-8')
(RUN_ROOT/'logs').mkdir(exist_ok=True)
(RUN_ROOT/'logs'/'run.log').write_text('Metrics run finished; validated 10 IR/VIS/fused IDs on same images.\n',encoding='utf-8')

summary.to_json(RUN_ROOT/'metrics'/'summary.json',orient='split',indent=2)
readme = f"""# SAGE diagnostic experiment 08/11/2026 — Lộc bé

- FMB/test repackaged samples: {len(IDS)} matched pairs. Source TAR: `{TAR_FILE.name}`.
- SAGE fused: `{FUSED_SOURCE}` (pretrained Student run from 05/11).
- Controlled comparisons: SAGE, Y_AVG, Y_MAX **on exactly the same 10 IDs**.
- **Real per-image metrics:** metrics/per_image.csv
- **Paired deltas:** metrics/paired_comparison.csv
- **Real example visuals:** figures/failure_case_candidates.png
- No current SOTA execution; compare literature separately with protocol caveats.
- No hyperparameter tuning on test images. IR_hot_ratio is a non-semantic intensity proxy.
- Results are diagnostic only, not evidence of a novel improved method.
"""
(RUN_ROOT/'README.md').write_text(readme,encoding='utf-8')
import shutil
archive=shutil.make_archive(str(RUN_ROOT),'zip',root_dir=RUN_ROOT)
print('Đã xuất:',archive)
print('Hãy tải ZIP này từ Kaggle Output và gửi lại để phân tích số liệu thật.')
